## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

In [2]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

while not Path("data/bubbles").exists():
    os.chdir("..")

BUBBLES_DIR = Path("data/bubbles")
VECTOR_DIR = Path("assets/vectorstores")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [3]:
MY_BUBBLE_FILE = "anti_suveranist.jsonl" 

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(bubble_path, lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: anti_suveranist
Texte: 50


,id,agent,text
0,yt_Tx8GhU2LeyI_UgwoWOyzF2UbPYnguUB4AaABAg,Anti-suveranist,Am toată încrederea că oameni ( de bine ) ca :...
1,yt_6_Hc2S02Duw_Ugytw6-BDQ2pA_Zi-TB4AaABAg,Anti-suveranist,Apropo de avalansa de troli ce se devarsa si a...
2,yt_bee6nXyzJ_E_UgxQ1N1kdP_MTx8B4K14AaABAg,Anti-suveranist,Aceasta nu este o emisiune....este o regizare ...
3,yt_bee6nXyzJ_E_Ugyjnx0utsCXEXc94q54AaABAg,Anti-suveranist,La pregatit bine Putin a investit bani in Guru...
4,yt_im3QoqSgfDo_UgwOL2VI3_toiLZSDqh4AaABAg,Anti-suveranist,"Eu am vorbit cu susținători de ai lui CG, îs d..."


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [4]:
texts = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
Am toată încrederea că oameni ( de bine ) ca : G Simion , Călinge , dna Găurilă ...... vor avea mare grijă să pună fie piedici , fie bețe-n roate astfel încât să rămanem sub tutela cremlinului


## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [ ]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)
#recomanda developer mode cand vb cu ionela si dragos

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.53it/s]

Număr texte: 50
Dimensiune embeddings: (50, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?

    50 de texte
- Câți vectori au fost generați?

    50 de vectori, câte unul pentru fiecare text
- Ce înseamnă a doua valoare din `embeddings.shape`?

    384 reprezintă dimensiunea fiecărui vector

## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [8]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: assets\vectorstores\anti_suveranist
Vectori în index: 50


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [9]:
# TODO student:
# index.faiss există: ...
# index.pkl există: ...
# index.ntotal este egal cu numărul de texte: ...
# 1. index.faiss există:
faiss_file = out_dir / "index.faiss"
print(f"index.faiss există: {faiss_file.exists()}")

# 2. index.pkl există:
pkl_file = out_dir / "index.pkl"
print(f"index.pkl există: {pkl_file.exists()}")

# 3. index.ntotal este egal cu numărul de texte:
# Verificăm dacă numărul de vectori din index (50) este egal cu numărul de texte din metadata (50)
numar_texte = len(metadata)
este_egal = index.ntotal == numar_texte

print(f"index.ntotal ({index.ntotal}) este egal cu numărul de texte ({numar_texte}): {este_egal}")

index.faiss există: True
index.pkl există: True
index.ntotal (50) este egal cu numărul de texte (50): True


## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În următorul continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [13]:
# Text nou introdus în aplicație

input_text = "Izolarea de partenerii occidentali și discursul naționalist distrug economia națională și ne lasă fără fonduri europene."

In [14]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

In [11]:
# query_vector

In [16]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.345
Text: Pai nu dorea Iohanis sa-l pună prim-ministru ce dracu faceți pe prostii.In primul tur.Georgescu a fost promovat sa distrugă curentul suveranist.Dobitocul asta de agent SIE face pe neștiutorul.

Rezultat 2
Scor: 0.335
Text: Cu respect vă rog ca de Ziua Națională să nu ne separați în 2 tabere: georgiști și auriști. Va fi și Simion la Alba Iulia, vă rog să nu faceți ca Șosoacă care a dezbinat miscarea suveranistă. Doar sistemul soroșist ar avea de câștigat.

Rezultat 3
Scor: 0.313
Text: Lumea nu vede că Aur este cu două fețe când cu turci când cu Rușii când cu Americanii

Rezultat 4
Scor: 0.304
Text: DIN PACATE RĂMÂN LA PAREREA LUI IURIE ROȘCA....CG MASON VADIT NEW AGE IST OPOZIȚIA CONTROLATĂ.....😢😢😢😢😢.... ESTE ATÂT DE FIRESC ....NU I VORBA DE GEORGESCU NEAPARAT... PÂNĂ LA CAPĂT ĂSTA ESTE POLITICUL SI CLOACA CE FUGE IREMEDIABIL DUPA O ...PUTERE CE NU EXISTĂ....ILUZII MATRIX CONSUMATOR DE ENERGIE A VISELOR A CELOR CARE NU SE MAI OPRESC DIN VISAT....😢

Rezultat

### TODO
Schimbă `input_text` cu o afirmație potrivită pentru agentul tău.
Rulează căutarea.
Notează:
- câte rezultate din 5 sunt relevante;
- dacă textele recuperate exprimă vocea agentului;
- dacă ai observat un text slab care ar trebui eliminat.

1. Câte rezultate din 5 sunt relevante?

    3 din 5 (Rezultatele 1, 3 și 5). Aceste comentarii combat direct liderii populiști sau discursul naționalist, acuzându-i de duplicitate („cu două fețe”), manipulare („agent SIE”) sau „propagandă cu vise comuniste”.

2. Dacă textele recuperate exprimă vocea agentului:

    Da. Aceste texte reflectă fidel vocea internautului critic, ironic și anti-populist. Totuși, Rezultatul 2 s-a strecurat greșit (aparține unui suveranist), fiind extras de algoritm doar pe baza cuvintelor-cheie similare.

3. Text slab observat ce ar trebui eliminat:

    Rezultatul 4. Este scris integral cu Caps Lock, plin de emoji-uri și axat pe teorii abstracte care nu au legătură cu politica („iluzii matrix”, „new age-ist”). Acest text va polua inutil contextul trimis către modelul de generare.